# Appendix: Performance & Risk Metrics

This notebook defines the core performance metrics used throughout the library — primarily Sharpe ratio and drawdown. These are foundational utilities depended on by `plot.py` (legend annotations) and all backtesting notebooks.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import pandas as pd

## Sharpe Ratio

The Sharpe ratio captures the gain of a strategy $E[\text{pnl}_t]$ for a level of risk $\text{Var}[\text{pnl}_t]$:

$$\text{sr} = \sqrt{N_{\text{periods per year}}} \times \frac{E[\text{pnl}_t]}{\sqrt{\text{Var}[\text{pnl}_t]}}$$

where $N_{\text{periods per year}}$ is an annualization factor (12 for monthly, 260 for business days, 365 for calendar days). The function auto-detects the frequency from the index.

## Drawdown

Drawdown measures the cumulative loss from the peak of a strategy's equity curve:

$$\text{dd}_t = \text{cumPnL}_t - \max_{s \le t} \text{cumPnL}_s$$

When `return_in_risk_unit=True`, the drawdown is normalized by a rolling estimate of volatility, making it comparable across strategies with different risk levels.

In [ ]:
%%writefile ../skfin/metrics.py
"""Performance and risk metrics."""

import numpy as np
import pandas as pd


def _test_monthly(df: pd.Series) -> bool:
    """Check if the series has monthly frequency."""
    return int(len(df) / len(df.asfreq("ME"))) == 1


def _test_bday(df: pd.Series) -> bool:
    """Check if the series has business-day frequency."""
    return int(len(df) / len(df.asfreq("B"))) == 1


def _test_day(df: pd.Series) -> bool:
    """Check if the series has calendar-day frequency."""
    return int(len(df) / len(df.asfreq("D"))) == 1


def sharpe_ratio(
    df: pd.Series,
    num_period_per_year: int | None = None,
    remove_zeros: bool = True,
) -> float:
    """Compute annualized Sharpe ratio.

    Args:
        df: PnL or returns series with a DatetimeIndex.
        num_period_per_year: Annualization factor. Auto-detected from index if None.
        remove_zeros: Replace zeros with NaN before computing (avoids deflating vol).

    Returns:
        Annualized Sharpe ratio, or NaN if frequency cannot be detected.
    """
    if num_period_per_year is None:
        if _test_monthly(df):
            num_period_per_year = 12
        if _test_bday(df):
            num_period_per_year = 260
        if _test_day(df):
            num_period_per_year = 365
        if num_period_per_year is None:
            return np.nan
    if remove_zeros:
        df = df.replace(0, np.nan)
    return df.mean() / df.std() * np.sqrt(num_period_per_year)


def drawdown(
    x: pd.Series,
    return_in_risk_unit: bool = True,
    window: int = 36,
    num_period_per_year: int = 12,
) -> pd.Series:
    """Compute drawdown from peak cumulative PnL.

    Args:
        x: PnL or returns series.
        return_in_risk_unit: Normalize by rolling volatility for cross-strategy comparison.
        window: Rolling window for volatility estimate (in periods).
        num_period_per_year: Annualization factor for the volatility normalization.

    Returns:
        Drawdown series (non-positive values; 0 at peaks).
    """
    dd = x.cumsum().sub(x.cumsum().cummax())
    if return_in_risk_unit:
        return dd.div(x.rolling(window).std().mul(np.sqrt(num_period_per_year)))
    return dd


In [ ]:
from skfin.metrics import sharpe_ratio, drawdown

## Usage Examples

### `sharpe_ratio`: monthly strategy

A strategy with positive mean return and monthly observations. The annualization factor (12) is auto-detected.

In [ ]:
np.random.seed(42)
monthly_pnl = pd.Series(
    np.random.normal(0.005, 0.03, 60),
    index=pd.date_range("2019-01-31", periods=60, freq="ME"),
)
print(f"Monthly Sharpe: {sharpe_ratio(monthly_pnl):.2f}")

### `sharpe_ratio`: daily strategy

Business-day frequency is detected automatically (annualization = 260).

In [ ]:
daily_pnl = pd.Series(
    np.random.normal(0.0003, 0.01, 520),
    index=pd.bdate_range("2022-01-03", periods=520),
)
print(f"Daily Sharpe: {sharpe_ratio(daily_pnl):.2f}")

### `sharpe_ratio`: explicit annualization

Override auto-detection when the frequency is non-standard (e.g. weekly data).

In [ ]:
weekly_pnl = pd.Series(
    np.random.normal(0.001, 0.02, 104),
    index=pd.date_range("2022-01-07", periods=104, freq="W-FRI"),
)
print(f"Weekly Sharpe (52 weeks/year): {sharpe_ratio(weekly_pnl, num_period_per_year=52):.2f}")

### `drawdown`: absolute vs risk-normalized

Compare raw drawdown (in return units) with risk-normalized drawdown (in volatility units).

In [ ]:
from skfin.plot import line

dd_abs = drawdown(monthly_pnl, return_in_risk_unit=False)
dd_risk = drawdown(monthly_pnl, return_in_risk_unit=True, window=12)

line(
    {"Absolute DD": dd_abs, "Risk-normalized DD": dd_risk},
    title="Drawdown: absolute vs risk-adjusted",
    sort=False,
    legend_sharpe_ratio=False,
)

### `drawdown`: comparing strategies

Risk-normalized drawdown makes strategies with different volatilities comparable.

In [ ]:
low_vol = pd.Series(
    np.random.normal(0.002, 0.01, 60),
    index=pd.date_range("2019-01-31", periods=60, freq="ME"),
)
high_vol = pd.Series(
    np.random.normal(0.004, 0.05, 60),
    index=pd.date_range("2019-01-31", periods=60, freq="ME"),
)

line(
    {
        "Low vol (risk-adj DD)": drawdown(low_vol, window=12),
        "High vol (risk-adj DD)": drawdown(high_vol, window=12),
    },
    title="Risk-normalized drawdown comparison",
    sort=False,
    legend_sharpe_ratio=False,
)